# BBC News scraper: headlines and descriptions

This notebook fetches the current BBC News homepage, extracts article headlines and descriptions, removes duplicates, previews the data, and saves it as a CSV file.

> Please respect BBC's terms of use, robots.txt, rate limits, and copyright. This example is for educational, low-volume scraping only.

In [10]:
from datetime import datetime, timezone
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

BBC_URL = "https://www.bbc.com/news"
OUTPUT_FILE = "bbc_news_today.csv"

response = requests.get(
    BBC_URL,
    headers={"User-Agent": "Mozilla/5.0 (compatible; educational-news-scraper/1.0)"},
    timeout=20,
)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")
retrieved_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
print(f"Fetched {BBC_URL} at {retrieved_at}")
print(f"HTTP status: {response.status_code}")

Fetched https://www.bbc.com/news at 2026-09-27T17:41:46+00:00
HTTP status: 200


In [11]:
def clean_text(value):
    return " ".join(value.split()) if value else ""


def find_description(heading):
    heading_text = clean_text(heading.get_text(" ", strip=True))

    # Search only until the next headline so one story cannot borrow another story's text.
    for element in heading.next_elements:
        if getattr(element, "name", None) in {"h2", "h3"}:
            break
        if getattr(element, "name", None) != "p":
            continue

        description = clean_text(element.get_text(" ", strip=True))
        if description and description != heading_text:
            return description

    return ""


records = []
seen_urls = set()

for heading in soup.select("h2, h3"):
    headline = clean_text(heading.get_text(" ", strip=True))
    link = heading.find_parent("a", href=True)

    if not headline or link is None:
        continue

    article_url = urljoin(BBC_URL, link["href"])
    if "/news/" not in article_url or article_url in seen_urls:
        continue

    seen_urls.add(article_url)
    records.append(
        {
            "headline": headline,
            "description": find_description(heading),
            "url": article_url,
            "retrieved_at_utc": retrieved_at,
        }
    )

news_df = pd.DataFrame(records).drop_duplicates(subset="url").reset_index(drop=True)

if news_df.empty:
    raise RuntimeError(
        "No BBC articles were found. The page markup may have changed, or access may be blocked."
    )

print(f"Extracted {len(news_df)} unique BBC articles")
print(f"Unique descriptions: {news_df['description'].nunique()} of {len(news_df)} rows")
news_df.head(10)

Extracted 28 unique BBC articles
Unique descriptions: 18 of 28 rows


,headline,description,url,retrieved_at_utc
0,Trump says men arrested over RAF Fairford inci...,Arrests made after three suspicious vehicles w...,https://www.bbc.com/news/live/c65y7nl29k0et,2026-09-27T17:41:46+00:00
1,Young people in China are fangirling over a pr...,China's economy and tech have developed at bre...,https://www.bbc.com/news/articles/cq70dgdndr46o,2026-09-27T17:41:46+00:00
2,Contentious Northern Ireland march at standsti...,"The march, which in the 1990s sparked some of ...",https://www.bbc.com/news/live/ckz6z4z57y5nt,2026-09-27T17:41:46+00:00
3,Two bodies found after avalanche hits Himalaya...,Rescuers are looking for at least 10 Nepalese ...,https://www.bbc.com/news/articles/cwjdvml9e897o,2026-09-27T17:41:46+00:00
4,The young women guiding tourists through India...,Astronomy tourism is giving young women in Han...,https://www.bbc.com/news/articles/cvp8d0nezkeko,2026-09-27T17:41:46+00:00
5,Watch: At the scene of counter-terrorism opera...,Five men have been arrested as counter-terrori...,https://www.bbc.com/news/videos/cxp84gq34n9zo,2026-09-27T17:41:46+00:00
6,The treasured 'eternal snow' on this tropical ...,Climate change is transforming Papua's snowy p...,https://www.bbc.com/news/articles/cm5ydvww0erdo,2026-09-27T17:41:46+00:00
7,Two mass shootings in South Africa leave 27 dead,The two attacks hours apart come as the countr...,https://www.bbc.com/news/articles/ck5ywnp075d1o,2026-09-27T17:41:46+00:00
8,Iranian minister says only negotiation can end...,,https://www.bbc.com/news/articles/cmvgyyw2jeego,2026-09-27T17:41:46+00:00
9,"'Scourge' of abuse must be rooted out, says Po...",,https://www.bbc.com/news/articles/cm5y5nj8ejj8o,2026-09-27T17:41:46+00:00


In [12]:
# Keep headline and description columns first for easier analysis.
news_df = news_df[["headline", "description", "url", "retrieved_at_utc"]]

try:
    news_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
except PermissionError:
    fallback_file = f"bbc_news_today_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    news_df.to_csv(fallback_file, index=False, encoding="utf-8-sig")
    OUTPUT_FILE = fallback_file
    print(f"The original CSV is locked, so the data was saved to {OUTPUT_FILE}")
else:
    print(f"Saved {len(news_df)} rows to {OUTPUT_FILE}")

news_df[["headline", "description"]].head(10)

The original CSV is locked, so the data was saved to bbc_news_today_20260927_224155.csv


,headline,description
0,Trump says men arrested over RAF Fairford inci...,Arrests made after three suspicious vehicles w...
1,Young people in China are fangirling over a pr...,China's economy and tech have developed at bre...
2,Contentious Northern Ireland march at standsti...,"The march, which in the 1990s sparked some of ..."
3,Two bodies found after avalanche hits Himalaya...,Rescuers are looking for at least 10 Nepalese ...
4,The young women guiding tourists through India...,Astronomy tourism is giving young women in Han...
5,Watch: At the scene of counter-terrorism opera...,Five men have been arrested as counter-terrori...
6,The treasured 'eternal snow' on this tropical ...,Climate change is transforming Papua's snowy p...
7,Two mass shootings in South Africa leave 27 dead,The two attacks hours apart come as the countr...
8,Iranian minister says only negotiation can end...,
9,"'Scourge' of abuse must be rooted out, says Po...",


## How this works

1. `requests.get()` downloads the page HTML.
2. `BeautifulSoup` parses the HTML document.
3. `h2` and `h3` elements provide likely article headlines.
4. The nearest article/card is searched for a paragraph description.
5. Article URLs are normalized, deduplicated, and exported to `bbc_news_today.csv`.

Run the cells from top to bottom each time you want a fresh snapshot. The output represents the page at retrieval time, not a guaranteed archive of every story published today. If BBC changes its HTML, inspect `response.text` and update the selectors.